In [1]:
#Section D: Practical Questions ( Try it yourself)

In [2]:
# D1. Relational -Build and Query
# Using sqlite3, create two tables: employees(employee_id PK, name, dept_id FK) 
# and departments(dept_id PK, dept_name). 
#Insert at least 2 departments and 3 employees. 
#Write a query that joins both tables to list each employee's name alongside their department name.

In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute("CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT)")
cur.execute("""CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
)""")

cur.executemany("INSERT INTO departments VALUES (?, ?)", [
    (1, "Engineering"), (2, "Sales")
])
cur.executemany("INSERT INTO employees (name, dept_id) VALUES (?, ?)", [
    ("Alice", 1), ("Bob", 1), ("Carol", 2)
])
conn.commit()

query = """
SELECT e.name, d.dept_name
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
"""
for row in cur.execute(query):
    print(row)

('Alice', 'Engineering')
('Bob', 'Engineering')
('Carol', 'Sales')


In [4]:
# D2. ACID - Transaction Rollback
# Using sqlite3, create a wallet(user, balance) 
# table. Simulate a transfer of 30 units from one user to another inside a transaction. 
# Deliberately raise an exception before committing, then roll back and print the table
# to confirm balances are unchanged.

In [6]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE wallet (user TEXT PRIMARY KEY, balance INTEGER)")
cur.executemany("INSERT INTO wallet VALUES (?, ?)", [("A", 100), ("B", 50)])
conn.commit()

try:
    cur.execute("BEGIN")
    cur.execute("UPDATE wallet SET balance = balance - 30 WHERE user = 'A'")
    cur.execute("UPDATE wallet SET balance = balance + 30 WHERE user = 'B'")
    raise Exception("failure before commit!")
    conn.commit()
except Exception as e:
    print("Error:", e)
    conn.rollback()

print(cur.execute("SELECT * FROM wallet").fetchall())
# Balances remain unchanged: [('A', 100), ('B', 50)]

Error: failure before commit!
[('A', 100), ('B', 50)]


In [7]:
# D3. Document Model - Variable Attributes
# Using Python dictionaries, create a small in-memory "collection" (list of dicts) 
# of at least 3 products from different categories (e.g., book, laptop, chair), 
# where each product has different fields beyond a shared sku and name. 
# Write a function that filters and returns only products that have a battery_life field.

In [8]:
products = [
    {"sku": "B001", "name": "Python Basics", "pages": 320, "author": "J. Doe"},
    {"sku": "L001", "name": "UltraBook Pro", "battery_life": "10h", "ram_gb": 16},
    {"sku": "C001", "name": "Office Chair", "material": "mesh", "color": "black"},
]

def filter_by_battery(catalog):
    return [p for p in catalog if "battery_life" in p]

print(filter_by_battery(products))


[{'sku': 'L001', 'name': 'UltraBook Pro', 'battery_life': '10h', 'ram_gb': 16}]


In [9]:
# D4. Key-Value Store - Simulated Cache
# Using a Python dict as a key-value store, write code to store 3 user sessions
# keyed by "session:<id>", then write a function get_session(session_id) 
# that returns the session or None if not found (simulating a cache miss).


cache = {
    "session:101": {"user": "Alice", "cart": ["sku1", "sku2"]},
    "session:102": {"user": "Bob", "cart": []},
    "session:103": {"user": "Carol", "cart": ["sku3"]},
}

def get_session(session_id):
    return cache.get(f"session:{session_id}", None)

print(get_session(101))   # returns Alice's session
print(get_session(999))   # None -> cache miss

{'user': 'Alice', 'cart': ['sku1', 'sku2']}
None


In [10]:
# D5. Eventual Consistency 
# Using two Python dicts (replicaA, replicaB) to simulate replicas,
# write a value to replicaA only, show that reading it from replicaB 
# returns nothing (stale/missing), then simulate a sync and show replicaB now has the value.
# Add a short comment explaining which BASE property this demonstrates.


replicaA = {}
replicaB = {}

# Write only to replicaA
replicaA["product:1"] = "In Stock"

print("Read from B:", replicaB.get("product:1"))  # None -> stale/missing

# Simulate async sync/replication delay
replicaB.update(replicaA)

print("Read from B after sync:", replicaB.get("product:1"))  # "In Stock"


Read from B: None
Read from B after sync: In Stock


In [13]:
# D6. Graph - Friends of Friends
# Using an adjacency list (dict of lists) representing a small social graph 
# with at least 4 users, write code to compute the friends-of-friends 
#for one user, excluding the user themself and their direct friends.

social_graph = {
    "Alice": ["Bob", "Carol"],
    "Bob": ["Alice", "Dave"],
    "Carol": ["Alice", "Dave"],
    "Dave": ["Bob", "Carol"],
}

def friends_of_friends(graph, user):
    direct_friends = set(graph.get(user, []))
    fof = set()
    for friend in direct_friends:
        fof.update(graph.get(friend, []))
    fof.discard(user)
    fof -= direct_friends
    return fof

print(friends_of_friends(social_graph, "Bob"))



{'Carol'}
